# Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for the short exercises in [10-statistical-foundations-and-ml-exercises.ipynb](10-statistical-foundations-and-ml-exercises.ipynb).
:::

## Exercise 1: Explore

Build a DataFrame of 60 stations with `elevation_m` uniform on [200, 3500] and `temp_celsius = 15 - 6.5 * elevation_m/1000 + noise` (noise standard deviation 1.5 °C, seeded). Make a seaborn regression plot of temperature against elevation and print `describe()`.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
elev = rng.uniform(200, 3500, 60)
temp = 15 - 6.5 * (elev / 1000) + rng.normal(0, 1.5, 60)
df = pd.DataFrame({"elevation_m": elev, "temp_celsius": temp})
sns.regplot(data=df, x="elevation_m", y="temp_celsius")
plt.show()
print(df.describe().round(2))

## Exercise 2: Correlation

Print the Pearson correlation matrix of the DataFrame and identify the sign of the temperature–elevation correlation.

In [ ]:
print(df.corr().round(3))   # temperature vs elevation is strongly negative (the lapse rate)

## Exercise 3: Fit a linear regression

Fit `LinearRegression` with `X = df[["elevation_m"]]` and `y = df["temp_celsius"]`. Print the recovered lapse rate in °C km⁻¹ and the intercept.

In [ ]:
from sklearn.linear_model import LinearRegression
X = df[["elevation_m"]]; y = df["temp_celsius"]
m = LinearRegression().fit(X, y)
print(round(m.coef_[0] * 1000, 2), "°C km^-1")
print(round(m.intercept_, 2), "°C")

## Exercise 4: Honest evaluation

Split the data (test size 0.3, fixed random state), fit on the training set, and report the test RMSE and R^2.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
m = LinearRegression().fit(Xtr, ytr)
p = m.predict(Xte)
print("RMSE:", round(root_mean_squared_error(yte, p), 3))
print("R^2: ", round(r2_score(yte, p), 3))

## Exercise 5: A pipeline with scaling

Build a `Pipeline` of `StandardScaler` followed by `LinearRegression`, fit it on the training set, and print its test R^2 (via `.score`).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
print(round(pipe.score(Xte, yte), 3))

## Exercise 6: Demonstrate overfitting

Fit a degree-8 polynomial pipeline on the training set and compare its R^2 on the training set with its R^2 on the test set. Comment on the gap.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
over = make_pipeline(PolynomialFeatures(degree=8), LinearRegression()).fit(Xtr, ytr)
print("train R^2:", round(over.score(Xtr, ytr), 3))
print("test  R^2:", round(over.score(Xte, yte), 3))
# a large train-test gap is overfitting: the flexible model memorised noise

## Exercise 7: Cross-validation

Use `cross_val_score` with 5 folds to estimate the R^2 of a plain `LinearRegression` on the full dataset, and print the mean and standard deviation across folds.

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")
print(round(scores.mean(), 3), round(scores.std(), 3))

## Exercise 8: Cluster and reduce the penguins data

Using the same `palmer_penguins.csv` data as the lecture (fetch it the same way), drop rows with missing measurements, then:

1. Cluster the penguins into 2 groups with `KMeans`, using `bill_length_mm` and `bill_depth_mm`. Print a crosstab of true species against cluster to see how well 2 clusters separate 3 species.
2. Reduce all four measurements (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`) to 2 components with `PCA`, after scaling them. Print the explained variance ratio.

In [ ]:
import pooch

penguins_path = pooch.retrieve(
    url="https://github.com/gse-unil/2026_MLEES_book/blob/main/data/part-I/palmer_penguins.csv",
    known_hash="sha256:f204db2c753b0937caac3cb35258562c14f073e4bbc76be24b4c51ce22767a93",
    fname="palmer_penguins.csv",
    path=pooch.os_cache("mlees"),
)

penguins = pd.read_csv(penguins_path).dropna(
    subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
).reset_index(drop=True)

from sklearn.cluster import KMeans
X = penguins[["bill_length_mm", "bill_depth_mm"]]
kmeans = KMeans(n_clusters=2, random_state=0, n_init=10).fit(X)
penguins["cluster"] = kmeans.labels_
print(pd.crosstab(penguins["species"], penguins["cluster"]))

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
Xs = StandardScaler().fit_transform(penguins[features])
pca = PCA(n_components=2).fit(Xs)
print(pca.explained_variance_ratio_.round(3))